In [109]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import pandas as pd
import os

In [110]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [111]:
class MyLabeledDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.annotations = pd.read_csv(csv_file)  # CSV with columns: filename,label
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_name = self.annotations.iloc[idx, 0]
        label = int(self.annotations.iloc[idx, 1])
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)
        
        return image, label


In [112]:
# Transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

# Paths
train_csv = "data/train/_annotations.csv"
train_dir = "/data/train/images"
valid_csv = "data/valid/_annotations.csv"
valid_dir = "/data/valid/images"
test_csv  = "data/test/_annotations.csv"
test_dir  = "/data/test/images"

# Datasets
train_data = MyLabeledDataset(train_csv, train_dir, transform=transform)
valid_data = MyLabeledDataset(valid_csv, valid_dir, transform=transform)
test_data  = MyLabeledDataset(test_csv,  test_dir, transform=transform)

# DataLoaders
trainloader = DataLoader(train_data, batch_size=32, shuffle=True)
validloader = DataLoader(valid_data, batch_size=32, shuffle=False)
testloader  = DataLoader(test_data,  batch_size=32, shuffle=False)

print(f"Training images: {len(train_data)}")
print(f"Validation images: {len(valid_data)}")
print(f"Test images: {len(test_data)}")


Training images: 4219
Validation images: 1213
Test images: 761


In [113]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)  
        self.pool = nn.MaxPool2d(2, 2)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 4 * 4, 128)  
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # (16, 16, 16)
        x = self.pool(F.relu(self.conv2(x)))  # (32, 8, 8)
        x = self.pool(F.relu(self.conv3(x)))  # (64, 4, 4)
        x = x.view(-1, 64 * 4 * 4)            # flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = CNNModel().to(device)
print(model)

CNNModel(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1024, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(root="data/train", transform=transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# Take one batch
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)
output = model(x)


Input shape: torch.Size([4, 3, 32, 32])
Output shape: torch.Size([4, 10])


In [115]:
criterion = nn.CrossEntropyLoss()  
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 40  # or whatever you want

for epoch in range(num_epochs):
    model.train()  # set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:  
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()          # reset gradients
        outputs = model(images)        # forward pass
        loss = criterion(outputs, labels)
        loss.backward()                # backward pass
        optimizer.step()               # update weights

        running_loss += loss.item() * images.size(0)  # accumulate loss

        # compute accuracy
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

Epoch 1/40, Loss: 0.8997, Accuracy: 0.6853
Epoch 2/40, Loss: 0.7480, Accuracy: 0.7194
Epoch 3/40, Loss: 0.6291, Accuracy: 0.7574


In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Make sure test images are in subfolders per class
test_dataset = datasets.ImageFolder(root="data/test", transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model.eval()  # switch to evaluation mode
with torch.no_grad():  # no gradient calculations needed
    correct = 0
    total = 0
    running_loss = 0.0

    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)  # optional
        running_loss += loss.item() * images.size(0)

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    test_loss = running_loss / len(test_loader.dataset)
    test_acc = correct / total

print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")
print({total})


Test Loss: 0.4099, Test Accuracy: 0.8274
{168}
